# Log
* trim each column
* remove the - in CID column
* normalize CNTRY column with standardize country name column, example USA to United States
* rename column

In [0]:

import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
loc_a101 = spark.table("workspace.bronze.erp_loc_a101")

loc_a101.display()

In [0]:
%sql
-- CHECK DISTINCT country
SELECT DISTINCT CNTRY
FROM workspace.bronze.erp_loc_a101

In [0]:
# Trim all string columns
for field in loc_a101.schema.fields:
    if isinstance(field.dataType, StringType):
        loc_a101 = loc_a101.withColumn(field.name, trim(col(field.name)))
     

In [0]:
#remove the "-" in id column
loc_a101 = loc_a101.withColumn(
    "CID",
    F.regexp_replace(F.col("CID"),"-","")
)

In [0]:
loc_a101 = loc_a101.withColumn(
    "CNTRY",
    F.when(F.col("CNTRY").isin("US","USA"),"United States").
    when(F.col("CNTRY")=='DE',"Germany").
    when((F.col("CNTRY")== "" )| F.col("CNTRY").isNull(),"Unknown").
    otherwise(F.col("CNTRY"))
)

In [0]:
loc_a101_cntry = loc_a101.select("CNTRY").distinct().show()

In [0]:
loc_a101.display()

In [0]:
rename_map_loc_a101 = {
    "CID": "customer_id",
    "CNTRY": "country"
}

for old_name, new_name in rename_map_loc_a101.items():
    loc_a101 = loc_a101.withColumnRenamed(old_name, new_name)

In [0]:
loc_a101.write.mode("overwrite").format('delta').saveAsTable("workspace.silver.erp_loc_a101")